In [1]:
from pathlib import Path
from datetime import time
import pandas as pd
import numpy as np

In [2]:
project_root = Path.cwd().resolve()
while not (project_root / "pyproject.toml").exists():
    if project_root == project_root.parent:
        raise FileNotFoundError("未找到项目根目录的 pyproject.toml")
    project_root = project_root.parent
print("项目根目录：", project_root)
raw_dir = project_root / "data" / "raw" / "chinese"
raw_dir.glob("*.csv")
processed_dir = project_root/'data'/'processed'/'chinese_return'

项目根目录： E:\anyfintech\research\in-progress\volatility-forecasting-based-on-DQN\volatility-forecasting


In [3]:
csv_files = list(raw_dir.glob("*.csv"))
print(f"找到 {len(csv_files)} 个 CSV 文件")

找到 40 个 CSV 文件


In [4]:
for i in range(len(csv_files)):
    file_path = csv_files[i]
    df = pd.read_csv(file_path)
    # Data Clean
    quote_cols = [
    "Open Bid Price",
    "Open Ask Price",
    "Close Bid Price",
    "Close Ask Price"
    ]
    
    ## Missing Value
    df["Open_missing"] = (
        df[["Open Bid Price", "Open Ask Price"]]
        .eq(0)
        .any(axis=1)
    )
    df["Close_missing"] = (
        df[["Close Bid Price", "Close Ask Price"]]
        .eq(0)
        .any(axis=1)
    )

    df[quote_cols] = df[quote_cols].mask(
        df[quote_cols].eq(0),
        np.nan,
    )

    df["Open"] = (
        df["Open Bid Price"]
        + df["Open Ask Price"]
    ) / 2

    df["Close"] = (
        df["Close Bid Price"]
        + df["Close Ask Price"]
    ) / 2
    
    df = df.sort_values(["Date", "Time"]).copy()
    df['Time'] = pd.to_datetime(df['Time']).dt.time
    mask = (
        df["Time"].between(time(9, 35), time(11, 30))
        | df["Time"].between(time(13, 5), time(15, 0))
    )
    df = df.loc[mask]
    
    is_session_start = df["Time"].isin({
        time(9, 35),
        time(13, 5)
    })

    df["required_price_missing"] = (
        df["Close"].isna().astype(int)
        + (
            is_session_start
            & df["Open"].isna()
        ).astype(int)
    )

    missing_count = (
        df.groupby("Date")["required_price_missing"]
        .transform("sum")
    )

    df["missing_ratio"] = missing_count / 50
    df = df.loc[df["missing_ratio"].le(0.05)].copy()
    df['session'] = np.where(df["Time"] <= time(11, 30),"AM","PM")
    
    bars_per_day = (
        df.groupby("Date")["Time"]
        .transform("nunique")
    )
    df = df.loc[bars_per_day.eq(48)].copy()
    
    df['Close_filled'] = df['Close'].fillna(df['Open'])
    df["Close_filled"] = (
        df.groupby(["Date", "session"])["Close_filled"].ffill()
    )
    df["Close_filled"] = (
        df.groupby(["Date", "session"])["Close_filled"].bfill()
    )
    df['Open_filled'] = df['Open'].fillna(df['Close_filled'])
    previous_close = (
        df.groupby(["Date", "session"])["Close_filled"]
        .shift(1)
    )
    df['Previous_Close'] = previous_close.fillna(df['Open_filled'])
    df["return_raw"] = np.log(
        df["Close_filled"] / df['Previous_Close']
    )
    df = df.sort_values(["Date", "Time"]).copy()
    df["bar_no"] = (
        df.groupby("Date")
        .cumcount()
    )
    df["hour_block"] = df["bar_no"] // 12
    df["squared_return_raw"] = df["return_raw"].pow(2)
    hourly_rv_raw = (
        df.groupby(["Date", "hour_block"])["squared_return_raw"]
        .sum()
    )
    hour_active = hourly_rv_raw.gt(0)
    day_active = (
        hour_active.groupby(level="Date")
        .agg(
            lambda values:
            len(values) == 4 and bool(values.all())
        )
    )
    active_dates = day_active.index[day_active]
    df_clean = (
        df.loc[df["Date"].isin(active_dates)]
        .sort_values(["Date", "Time"])
        .reset_index(drop=True)
        .copy()
    )
    df = df_clean[['Date', 'Time','return_raw']].copy()
    df.to_csv(processed_dir/file_path.name)


C:\Users\Tan\AppData\Local\Temp\ipykernel_5040\3052816740.py:40: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Time'] = pd.to_datetime(df['Time']).dt.time
C:\Users\Tan\AppData\Local\Temp\ipykernel_5040\3052816740.py:40: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Time'] = pd.to_datetime(df['Time']).dt.time
C:\Users\Tan\AppData\Local\Temp\ipykernel_5040\3052816740.py:40: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Time'] = pd.to_datetime(df['Time']).dt.time
C:\Users\Tan\AppData\Local\Temp\ipykernel_5040\3052816740.py:40: UserWarning: Could not infer format, s